In [1]:
!pip install -q google-generativeai faiss-cpu pypdf




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 18.4 MB/s eta 0:00:00


In [2]:

import google.generativeai as genai
from pypdf import PdfReader
import faiss
import numpy as np
from google.colab import files



/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
genai.configure(api_key="GEMINI_API_KEY")


In [4]:
uploaded = files.upload()


Saving class 11 chemistry (1).pdf to class 11 chemistry (1).pdf


In [5]:
documents = []

for file_name in uploaded.keys():
    reader = PdfReader(file_name)
    for page in reader.pages:
        text = page.extract_text()
        if text:
            documents.append(text)


In [6]:
chunk_size = 500
chunks = []

for doc in documents:
    for i in range(0, len(doc), chunk_size):
        chunks.append(doc[i:i+chunk_size])


In [10]:
embeddings = []

for chunk in chunks:
    emb = genai.embed_content(
        model="models/embedding-001",
        content=chunk
    )
    embeddings.append(emb["embedding"])

embeddings = np.array(embeddings).astype("float32")

In [10]:
dimension = len(embeddings[0])

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)


In [10]:
query = input("Ask a question: ")

query_embedding = genai.embed_content(
    model="models/embedding-001",
    content=query
)["embedding"]

query_embedding = np.array([query_embedding]).astype("float32")

D, I = index.search(query_embedding, k=3)

context = " ".join([chunks[i] for i in I[0]])


In [10]:
model = genai.GenerativeModel("gemini-pro")

prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{query}
"""

response = model.generate_content(prompt)

print(response.text)
